# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

## Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display high-level dataset metadata
print("Loaded Dataset Metadata:")
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", getattr(dataset.metadata, 'version', ''))
print("License:", getattr(dataset.metadata, 'license', ''))
print("Published On:", getattr(dataset.metadata, 'datePublished', ''))

## 2. Data Overview

Review available record sets, fields, and their unique `@id`s.

The Croissant dataset structure is organized as follows:
- **Record Sets**: Each `RecordSet` represents a conceptual table or entity in the dataset, identified by a unique `@id`.
- **Fields/Columns**: Each field (column) of a `RecordSet` is uniquely identified by its `@id`.

Let's list the available `RecordSet` (by `@id`) and their columns.

In [ ]:
# Get all record sets defined in the Croissant schema by their @id
record_sets = [rs['@id'] for rs in dataset.metadata.dict().get('recordSet', [])]

print(f"Found {len(record_sets)} record sets:")
for rs in dataset.metadata.dict().get('recordSet', []):
    rs_id = rs['@id']
    print(f"- RecordSet @id: {rs_id}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"    Columns/Field @ids: {[f['@id'] for f in fields]}")

## 3. Data Extraction

Let's extract one or more record sets as Pandas DataFrames for analysis.

**Note:**
- All record set and field references use their `@id`.
- Identify a record set of interest (see above overview, typically beginning with `cr:` or similar).
- We'll load all available record sets into a dictionary of DataFrames.

In [ ]:
# If the dataset has record sets, extract their data
# As of 2024-07, this dataset may have a single main record set
all_dfs = {}

if not record_sets:
    print("No record sets found in metadata (Croissant schema's 'recordSet' field is empty).")
    # Depending on dataset version, sometimes record sets are not referenced from 'recordSet' field.
    # Try listing possible implicit record set IDs
    # For mlcroissant >=0.12, list available record sets via `dataset.record_sets()`
    try:
        available = dataset.record_sets()
        print("Available record sets as discovered (via dataset.record_sets()):", available)
        record_sets = available
    except Exception as e:
        print("Could not discover record sets automatically:", e)
else:
    print("Using record sets found in metadata.")

# Load records for each record set and store as DataFrame
for rs_id in record_sets:
    print(f"\nLoading records from {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
        all_dfs[rs_id] = df
        display_cols = df.columns.tolist()
        # Display top records
        display(df.head())
    else:
        print(f"No records found for {rs_id}.")

# If only one DataFrame, pick it for EDA by default
main_rs_id = record_sets[0] if record_sets else None
main_df = all_dfs.get(main_rs_id) if main_rs_id else None

## 4. Exploratory Data Analysis (EDA)

Let's practice common EDA using a numeric field from the main table. We'll:
- Filter records based on a numeric field (e.g., GAD-7, PHQ-9, or PSQ scores)
- Normalize that field (z-score)
- Group by a demographic, such as `level_of_education` or `gender`

> **All column/field references use their `@id`.** If you do not know the field `@id`, refer to the output above.

In [ ]:
# Example: EDA on GAD-7 Score
# You may need to adjust these IDs based on the printed columns in the previous cell.

if main_df is not None:
    # Try to auto-detect some typical field ids for scores
    gad7_candidates = [col for col in main_df.columns if 'gad' in col.lower() and 'score' in col.lower()]
    phq9_candidates = [col for col in main_df.columns if 'phq' in col.lower() and 'score' in col.lower()]
    psq_candidates = [col for col in main_df.columns if 'psq' in col.lower() and 'score' in col.lower()]

    # Choose the first available
    numeric_field = None
    for test in [gad7_candidates, phq9_candidates, psq_candidates]:
        if test:
            numeric_field = test[0]
            break
    if numeric_field is None:
        # Try any numeric column
        candidates = main_df.select_dtypes(include=['number']).columns.tolist()
        numeric_field = candidates[0] if candidates else None
    
    if numeric_field:
        print(f"Selected numeric field for EDA: {numeric_field}")
        # Example filter: Only show records with gad7 > threshold
        threshold = 10
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try to group by 'level_of_education' or 'gender', falling back to first categorical field
        possible_group_fields = [col for col in main_df.columns for key in ['education', 'gender'] if key in col.lower()]
        group_field = possible_group_fields[0] if possible_group_fields else None

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by '{group_field}':")
            display(grouped_df)
        else:
            print('No suitable group field for grouping analysis found.')
    else:
        print('No suitable numeric field found for EDA.')
else:
    print('No main DataFrame loaded; skipping EDA section.')

## 5. Visualization

Let's show a histogram of a selected score (e.g., GAD-7) and a boxplot grouped by a demographic such as education or gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Histogram
    sns.histplot(main_df[numeric_field].dropna(), bins=20, ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field}")
    axes[0].set_xlabel(numeric_field)

    # Boxplot by group (if possible)
    if group_field and group_field in main_df.columns:
        sns.boxplot(data=main_df, x=group_field, y=numeric_field, ax=axes[1])
        axes[1].set_title(f"{numeric_field} by {group_field}")
        axes[1].set_xlabel(group_field)
        axes[1].set_ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
    else:
        plt.sca(axes[1])
        plt.text(0.5, 0.5, 'No suitable group field for boxplot', ha='center', va='center')
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to explore the FAIR² Mental Health Survey Dataset from Kilifi County using `mlcroissant`.
- We loaded the dataset via its Croissant schema.
- Explored metadata, record sets, and fields by unique `@id`.
- Loaded the main data table and conducted sample exploratory data analysis, including normalization and grouping.
- Visualized score distributions and demographic splits.

For further analysis, consult the Croissant metadata for exact field definitions and consider advanced analysis or ML.